# Population Estimation: Lincoln-Petersen Index

**Formula:** $\hat{N} = \frac{N_1 \times N_2}{N_b}$

- $N_1$ = number of unique individuals captured on Day 1
- $N_2$ = number of unique individuals captured on Day 2
- $N_b$ = number of individuals captured on **both** days

Each cluster in `lca_annots.json` represents one individual animal.

In [100]:
import json
from datetime import datetime, date, timezone
from collections import defaultdict
import math

ANNOTS_PATH = '/fs/ess/PAS2136/ggr_data/results/GGR2024_fixed_encounter/lca/lca_annots.json'
# ANNOTS_PATH = '/fs/ess/PAS2136/ggr_data/wbia/GGR2018/gt_annots_final_gz_identified.json'
# ANNOTS_PATH = '/fs/ess/PAS2136/ggr_data/wbia/GGR2018/gt_annots_final_laikipia.json'
# ANNOTS_PATH = '/fs/ess/PAS2136/ggr_data/results/GGR2018_Laikipia/lca/lca_annots_intersection.json'
#ANNOTS_PATH = '/fs/ess/PAS2136/ggr_data/results/GGR2024_fixed/lca/lca_annots.json'
# ANNOTS_PATH = '/fs/ess/PAS2136/ggr_data/results/GGR2018_Laikipia/lca/lca_annots_shared.json'
# ANNOTS_PATH = '/fs/ess/PAS2136/ggr_data/wbia/GGR2018/gt_annots_final_gz_identified_shared.json'
# ANNOTS_PATH = '/fs/ess/PAS2136/ggr_data/results/GGR2018_ilan_intersect_wbia/final_annots_bridged.json'
# ANNOTS_PATH= '/fs/ess/PAS2136/ggr_data/results/GGR2018_ilan_intersect_wbia/lca/lca_annots_human_consistent.json'

with open(ANNOTS_PATH) as f:
    data = json.load(f)

images = {img['uuid']: img for img in data['images']}
annots = data['annotations']

print(f'Total annotations: {len(annots)}')
print(f'Total images: {len(images)}')

Total annotations: 4197
Total images: 3778


In [101]:
import json
from collections import Counter
import pandas as pd

BASE_FIXED = '/fs/ess/PAS2136/ggr_data/results/GGR2024_fixed'
BASE = '/fs/ess/PAS2136/ggr_data/results/GGR2024_fixed_encounter'

pipeline_steps = [
    ('Detection',              f'{BASE_FIXED}/detector/img_annots.json'),
    ('Species Identification', f'{BASE_FIXED}/species_identifier/si_annots.json'),
    ('Viewpoint Classification', f'{BASE_FIXED}/viewpoint_classifier/vc_annots.json'),
    ('IA Classification',      f'{BASE_FIXED}/ia_classifier/ia_annots.json'),
    ('IA Filtering',           f'{BASE_FIXED}/ia_classifier/ia_filtered_annots.json'),
    ('ID Region',              f'{BASE}/id_region/id_regions.json'),
    ('ID Region Filtering',    f'{BASE}/id_region/id_regions_filtered.json'),
    ('Encounter Grouping',     f'{BASE}/encounter_grouping/eg_annots.json'),
    ('Intra LCA',              f'{BASE}/encounter_lca/lca_annots.json'),
    ('Representative Selection', f'{BASE}/representative/representative_annots.json'),
    ('Inter LCA',              f'{BASE}/inter_lca/lca_annots.json'),
    ('Forward Clustering',     f'{BASE}/lca/lca_annots.json'),
]

rows = []
for step_name, path in pipeline_steps:
    try:
        with open(path) as f:
            data = json.load(f)
        annots_list = data.get('annotations', [])
        images_list = data.get('images', [])
        n_annots = len(annots_list)
        n_images = len(images_list)
        
        # Clusters (individuals)
        clusters = set(a.get('cluster_id') for a in annots_list if a.get('cluster_id') is not None)
        cluster_str = len(clusters) if clusters else '-'
        
        note = ''
        if step_name == 'Encounter Grouping':
            encounters = set(img.get('occurence_id') for img in images_list if img.get('occurence_id') is not None)
            note = f'{len(encounters)} encounters'
        elif step_name == 'Intra LCA':
            encounter_ids = set(a.get('encounter_id') for a in annots_list if a.get('encounter_id') is not None)
            cluster_str = len(encounter_ids)
            note = f'{len(encounter_ids)} intra-encounter clusters (1 representative each)'
        elif step_name == 'Representative Selection':
            reps = [a for a in annots_list if a.get('representative') == True]
            encounter_ids = set(a.get('encounter_id') for a in annots_list if a.get('encounter_id') is not None)
            cluster_str = len(encounter_ids)
            note = f'{len(reps)} representatives selected for inter LCA'
        elif step_name == 'Inter LCA':
            cluster_counts = Counter(a.get('cluster_id') for a in annots_list if a.get('cluster_id') is not None)
            singletons = sum(1 for cnt in cluster_counts.values() if cnt == 1)
            note = f'{len(cluster_counts)} unique individuals, {singletons} singletons'
        elif step_name == 'Forward Clustering':
            cluster_counts = Counter(a.get('cluster_id') for a in annots_list if a.get('cluster_id') is not None)
            singletons = sum(1 for cnt in cluster_counts.values() if cnt == 1)
            note = f'{len(cluster_counts)} unique individuals, {singletons} singletons'
        
        rows.append({'Step': step_name, 'Annotations': n_annots, 'Images': n_images, 'Clusters': cluster_str, 'Notes': note})
    except FileNotFoundError:
        rows.append({'Step': step_name, 'Annotations': '(not found)', 'Images': '', 'Clusters': '', 'Notes': ''})

df = pd.DataFrame(rows)
df

,Step,Annotations,Images,Clusters,Notes
0,Detection,(not found),,,
1,Species Identification,21684,11517,-,
2,Viewpoint Classification,20719,11165,-,
3,IA Classification,20719,11165,-,
4,IA Filtering,8188,7309,-,
5,ID Region,8188,7309,-,
6,ID Region Filtering,8006,7226,-,
7,Encounter Grouping,4197,3778,-,245 encounters
8,Intra LCA,4197,3778,1380,1380 intra-encounter clusters (1 representativ...
9,Representative Selection,4197,3778,1380,1380 representatives selected for inter LCA


## 0. Pipeline annotation statistics

## 1. Identify census days and assign individuals to days

In [102]:
# Map each annotation to its date (UTC)
day_clusters = defaultdict(set)        # date -> set of cluster_ids
day_annot_count = defaultdict(int)     # date -> number of annotations
day_images = defaultdict(set)          # date -> set of image_uuids

for a in annots:
    img = images[a['image_uuid']]
    dt = datetime.fromtimestamp(img['timestamp'], tz=timezone.utc)
    d = dt.date()
    day_clusters[d].add(a['cluster_id'])
    day_annot_count[d] += 1
    day_images[d].add(a['image_uuid'])

print('Day breakdown (UTC):')
print(f'{"Date":<15} {"Day":<12} {"Images":>10} {"Annotations":>12} {"Individuals":>12}')
print('-' * 65)
for d in sorted(day_clusters.keys()):
    print(f'{str(d):<15} {d.strftime("%A"):<12} {len(day_images[d]):>10} '
          f'{day_annot_count[d]:>12} {len(day_clusters[d]):>12}')


Day breakdown (UTC):
Date            Day              Images  Annotations  Individuals
-----------------------------------------------------------------
2024-01-27      Saturday           2082         2325          620
2024-01-28      Sunday             1696         1872          516


## 2. Lincoln-Petersen estimate using the two main census days

In [103]:
# The two main GGR census days
DAY1 = date(2024, 1, 27)
DAY2 = date(2024, 1, 28)

# DAY1 = date(2018, 1, 27)
# DAY2 = date(2018, 1, 28)

N1_set = day_clusters[DAY1]  # individuals seen on Day 1
N2_set = day_clusters[DAY2]  # individuals seen on Day 2
Nb_set = N1_set & N2_set     # individuals seen on both days

N1 = len(N1_set)
N2 = len(N2_set)
Nb = len(Nb_set)

A1 = day_annot_count[DAY1]
A2 = day_annot_count[DAY2]

print(f'Day 1 ({DAY1}): {A1} annotations, N1 = {N1} unique individuals')
print(f'Day 2 ({DAY2}): {A2} annotations, N2 = {N2} unique individuals')
print(f'Both days:      Nb = {Nb} recaptured individuals')
print()

Day 1 (2024-01-27): 2325 annotations, N1 = 620 unique individuals
Day 2 (2024-01-28): 1872 annotations, N2 = 516 unique individuals
Both days:      Nb = 256 recaptured individuals



In [104]:
# Lincoln-Petersen estimate
N_hat = (N1 * N2) / Nb
print(f'Lincoln-Petersen estimate: N = N1 * N2 / Nb = {N1} * {N2} / {Nb} = {N_hat:.0f}')
print()

Lincoln-Petersen estimate: N = N1 * N2 / Nb = 620 * 516 / 256 = 1250



## 3. Chapman correction (less biased for small samples)

$\hat{N}_C = \frac{(N_1 + 1)(N_2 + 1)}{N_b + 1} - 1$

In [105]:
N_chapman = ((N1 + 1) * (N2 + 1)) / (Nb + 1) - 1
print(f'Chapman estimate: N_c = {N_chapman:.0f}')
print()

Chapman estimate: N_c = 1248



## 4. Confidence interval (approximate 95% CI)

Variance (Chapman): $\text{Var}(\hat{N}_C) = \frac{(N_1+1)(N_2+1)(N_1-N_b)(N_2-N_b)}{(N_b+1)^2(N_b+2)}$

In [106]:
var_chapman = ((N1 + 1) * (N2 + 1) * (N1 - Nb) * (N2 - Nb)) / ((Nb + 1)**2 * (Nb + 2))
se = math.sqrt(var_chapman)
ci_lower = N_chapman - 1.96 * se
ci_upper = N_chapman + 1.96 * se

print(f'Standard error: {se:.1f}')
print(f'95% CI: [{ci_lower:.0f}, {ci_upper:.0f}]')

Standard error: 42.2
95% CI: [1165, 1331]


## 5. Summary

In [107]:
# Totals across both days
total_observed = len(N1_set | N2_set)
only_day1 = len(N1_set - N2_set)
only_day2 = len(N2_set - N1_set)

I1 = len(day_images[DAY1])
I2 = len(day_images[DAY2])
I_total = len(day_images[DAY1] | day_images[DAY2])
A1 = day_annot_count[DAY1]
A2 = day_annot_count[DAY2]
A_total = A1 + A2

print('=' * 60)
print('POPULATION ESTIMATE SUMMARY')
print('=' * 60)
print(f'{"":20s} {"Day 1":>10s} {"Day 2":>10s} {"Total":>10s}')
print(f'{"Images":20s} {I1:>10d} {I2:>10d} {I_total:>10d}')
print(f'{"Annotations":20s} {A1:>10d} {A2:>10d} {A_total:>10d}')
print(f'{"Individuals":20s} {N1:>10d} {N2:>10d} {total_observed:>10d}')
print('-' * 60)
print(f'Recaptured (Nb):               {Nb}')
print(f'Seen only Day 1:               {only_day1}')
print(f'Seen only Day 2:               {only_day2}')
print(f'Recapture rate:                {Nb/N1:.1%} of Day 1 seen again on Day 2')
print('-' * 60)
print(f'Lincoln-Petersen estimate:     {N_hat:.0f}')
print(f'Chapman estimate:              {N_chapman:.0f}')
print(f'95% CI:                        [{ci_lower:.0f}, {ci_upper:.0f}]')
print('=' * 60)


POPULATION ESTIMATE SUMMARY
                          Day 1      Day 2      Total
Images                     2082       1696       3778
Annotations                2325       1872       4197
Individuals                 620        516        880
------------------------------------------------------------
Recaptured (Nb):               256
Seen only Day 1:               364
Seen only Day 2:               260
Recapture rate:                41.3% of Day 1 seen again on Day 2
------------------------------------------------------------
Lincoln-Petersen estimate:     1250
Chapman estimate:              1248
95% CI:                        [1165, 1331]
